# Boston Housing: regresión lineal

Este notebook construye un modelo de regresión lineal para estimar el valor medio de las viviendas (`MEDV`) a partir de varias características.

El procedimiento utiliza dos niveles de evaluación:

1. `train_test_split` reserva un conjunto de prueba que se consulta una sola vez al final.
2. `KFold` evalúa la estabilidad del modelo usando únicamente el conjunto de entrenamiento.

Esta combinación resulta adecuada cuando hay pocos datos, porque la validación cruzada aprovecha varias veces las observaciones de entrenamiento sin contaminar la prueba final.

## 1. Bibliotecas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split

RANDOM_STATE = 42
pd.set_option("display.max_columns", None)

## 2. Carga de datos con pandas

El notebook busca primero `boston.csv` en el mismo directorio. Después normaliza los nombres de las columnas a mayúsculas para mantener una notación uniforme.

In [ ]:
datos = pd.read_csv("./boston.csv")
datos.columns = datos.columns.str.strip().str.upper()

datos.head()

Selección de datos de entrada del algoritmo

In [ ]:
# Variable que se desea predecir
OBJETIVO = "MEDV"

# Todas las demás columnas serán características de entrada
CARACTERISTICAS = "CRIM"

# Construir entradas X y salida y
X = datos[[CARACTERISTICAS]]
y = datos[OBJETIVO]

print("Dimensión de X:", X.shape)
print("Dimensión de y:", y.shape)

## 3. Separación entre entrenamiento y prueba

El conjunto de prueba representa datos no utilizados durante el ajuste ni durante la selección del procedimiento. El parámetro `random_state` hace reproducible la división.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

print(f"Entrenamiento: {len(X_train)} observaciones")
print(f"Prueba: {len(X_test)} observaciones")

## 4. Modelo de regresión lineal

`LinearRegression` estima un intercepto y un coeficiente para cada característica. Cada coeficiente expresa el cambio esperado en `MEDV` cuando la característica aumenta una unidad y las demás permanecen constantes.

In [ ]:
modelo = LinearRegression()

## 5. Validación cruzada KFold sobre entrenamiento

`KFold` divide el conjunto de entrenamiento en cinco partes. En cada iteración utiliza cuatro partes para ajustar y una para validar. Se reportan la media y la desviación estándar para observar tanto el desempeño como su variabilidad.

El conjunto de prueba no participa en este proceso.

In [ ]:
# Configuración de la validación cruzada
kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Evaluación del modelo
resultados = cross_validate(
    estimator=modelo,
    X=X_train,
    y=y_train,
    cv=kfold,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "MSE": "neg_mean_squared_error",
        "R2": "r2"
    }
)

# Resultados de cada partición
metricas_cv = pd.DataFrame({
    "MAE": -resultados["test_MAE"],
    "RMSE": np.sqrt(-resultados["test_MSE"]),
    "R2": resultados["test_R2"]
})

metricas_cv.index = np.arange(1, len(metricas_cv) + 1)
metricas_cv.index.name = "Partición"

display(metricas_cv.round(3))

In [ ]:
resumen_cv = metricas_cv.agg(["mean", "std"]).rename(
    index={"mean": "Media", "std": "Desviación estándar"}
)
display(resumen_cv.round(3))

## 6. Ajuste final y evaluación sobre prueba

Después de comprobar el comportamiento mediante KFold, el modelo se ajusta con todas las observaciones de entrenamiento. Solo entonces se calcula el error sobre el conjunto de prueba reservado.

In [ ]:
def calcular_metricas(y_real, y_pred):
    return pd.Series({
        "MAE": mean_absolute_error(y_real, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_real, y_pred)),
        "R2": r2_score(y_real, y_pred),
    })

# entrena el modelo
modelo.fit(X_train, y_train)

y_pred_train = modelo.predict(X_train)
y_pred_test = modelo.predict(X_test)

metricas_finales = pd.DataFrame({
    "Entrenamiento": calcular_metricas(y_train, y_pred_train),
    "Prueba": calcular_metricas(y_test, y_pred_test),
})

display(metricas_finales.round(3))

## 7. Parámetros del modelo

In [ ]:
print(f"b_0 = {modelo.intercept_:.4f}")

for i, coeficiente in enumerate(modelo.coef_, start=1):
    print(f"b_{i} = {coeficiente:.4f}")

Los coeficientes tienen unidades diferentes. Por tanto, su magnitud directa no constituye una medida suficiente de importancia. Su interpretación debe considerar la unidad de cada característica y la posible correlación entre variables.

## 8. Visualización de predicciones y residuos

In [ ]:
# Datos ordenados para dibujar la predicción
X_total = pd.concat([X_train, X_test]).sort_values(
    by=CARACTERISTICAS
)

y_prediccion = modelo.predict(X_total)

plt.figure(figsize=(10, 5))

# Datos de entrenamiento
plt.scatter(
    X_train[CARACTERISTICAS],
    y_train,
    color="blue",
    label="Entrenamiento"
)

# Datos de validación
plt.scatter(
    X_test[CARACTERISTICAS],
    y_test,
    color="orange",
    label="Validación"
)

# Predicción del modelo
plt.plot(
    X_total[CARACTERISTICAS],
    y_prediccion,
    color="red",
    linewidth=2,
    label="Predicción del modelo"
)

plt.xlabel(CARACTERISTICAS)
plt.ylabel(OBJETIVO)
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
def graficar_diagnostico(y_real, y_pred):
    residuos = y_real.to_numpy() - y_pred
    limite_min = min(y_real.min(), y_pred.min())
    limite_max = max(y_real.max(), y_pred.max())

    fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))

    ejes[0].scatter(y_real, y_pred, alpha=0.75)
    ejes[0].plot([limite_min, limite_max], [limite_min, limite_max], "--", color="black")
    ejes[0].set(
        title="Valores reales y estimados",
        xlabel="MEDV real",
        ylabel="MEDV estimado",
    )

    ejes[1].scatter(y_pred, residuos, alpha=0.75)
    ejes[1].axhline(0, linestyle="--", color="black")
    ejes[1].set(
        title="Residuos del conjunto de prueba",
        xlabel="MEDV estimado",
        ylabel="Residuo",
    )

    for eje in ejes:
        eje.grid(alpha=0.25)

    plt.tight_layout()
    plt.show()


graficar_diagnostico(y_test, y_pred_test)

## 9. Interpretación de resultados

Para concluir el análisis, responda:

1. ¿Qué tan próximos son el RMSE medio de KFold y el RMSE del conjunto de prueba?
2. ¿La desviación estándar entre particiones indica un comportamiento estable?
3. ¿Existe una diferencia importante entre el error de entrenamiento y el de prueba?
4. ¿Los residuos se distribuyen alrededor de cero o muestran una estructura?
5. ¿Qué limitaciones tiene interpretar los coeficientes cuando las características están correlacionadas?

Una evaluación favorable requiere un error de prueba razonable, resultados semejantes entre las particiones y residuos sin patrones evidentes.